In [5]:
# ==============================================================================
# Updated Full R Script with Additional Filters and Expanded iPSC Terms
# ==============================================================================

library(GEOquery)
library(GEOmetadb)
library(DBI)
library(RSQLite)
library(dplyr)
library(stringr)

options(timeout = 3600)

# --- Function: Safe Online GEOquery Search ---
search_with_geoquery_safe <- function(search_terms, max_results = 20) {
  all_results <- list()
  
  for (term in search_terms) {
    # Expanded to prioritize RNA-seq explicitly
    query <- paste0('"', term, '"[All Fields] AND Homo sapiens[ORGN] AND GSE[Entry Type] AND RNA-seq[DataSet Type]')
    cat("Searching GEO for:", term, "\n")
    
    tryCatch({
      search_res <- getGEO(query)
      
      if (is.list(search_res)) {
        df_res <- as.data.frame(search_res[[1]])
      } else if (is.data.frame(search_res)) {
        df_res <- search_res
      } else {
        df_res <- NULL
      }
      
      if (!is.null(df_res) && nrow(df_res) > 0) {
        if (length(all_results) > 0) {
          common_cols <- intersect(names(all_results[[1]]), names(df_res))
          df_res <- df_res[, common_cols, drop = FALSE]
          all_results <- lapply(all_results, function(x) x[, common_cols, drop = FALSE])
        }
        df_res$search_term <- term
        all_results[[length(all_results) + 1]] <- head(df_res, max_results)
      } else {
        cat("No valid results for:", term, "\n")
      }
    }, error = function(e) {
      cat("Error searching for", term, ":", e$message, "\n")
    })
    Sys.sleep(1)
  }
  
  if (length(all_results) > 0) {
    return(bind_rows(all_results))
  } else {
    return(data.frame())
  }
}

# --- Function: Local GEOmetadb Search for RNA-seq only ---
search_geo_cell_types <- function(cell_types, min_samples = 5, geo_sqlite_file = "GEOmetadb.sqlite") {
  
  if (!file.exists(geo_sqlite_file)) {
    cat("Error: GEOmetadb.sqlite not found locally. Please download it first.\n")
    return(data.frame())
  }
  
  con <- dbConnect(SQLite(), geo_sqlite_file)
  results <- data.frame()
  
  for (cell_type in cell_types) {
    sql_query <- paste(
      "SELECT DISTINCT gse.gse, gse.title, gse.summary, gpl.title as platform,",
      "COUNT(DISTINCT gsm.gsm) as samples, gse.pubmed_id",
      "FROM gse",
      "JOIN gse_gsm ON gse_gsm.gse = gse.gse",
      "JOIN gsm ON gse_gsm.gsm = gsm.gsm",
      "JOIN gse_gpl ON gse.gse = gse_gpl.gse",
      "JOIN gpl ON gse_gpl.gpl = gpl.gpl",
      "WHERE (gse.title LIKE '%", cell_type, "%' OR gse.summary LIKE '%", cell_type, "%' OR gsm.title LIKE '%", cell_type, "%')",
      "AND gpl.organism LIKE '%Homo sapiens%'",
      "AND gpl.technology LIKE '%high-throughput sequencing%'",
      "AND gse.gse LIKE 'GSE%'",
      "GROUP BY gse.gse",
      "HAVING samples >= ", min_samples,
      "ORDER BY samples DESC"
    )
    
    cat("Searching for cell type:", cell_type, "\n")
    result <- dbGetQuery(con, sql_query)
    
    if (nrow(result) > 0) {
      result$search_term <- cell_type
      results <- rbind(results, result)
    }
  }
  
  dbDisconnect(con)
  return(results)
}

# --- Validation function remains unchanged ---
validate_geo_dataset <- function(gse_accession) {
  tryCatch({
    cat("Validating ", gse_accession, "...\n")
    gse <- getGEO(gse_accession, GSEMatrix = TRUE, getGPL = FALSE)
    if (length(gse) > 0) {
      gse_data <- gse[[1]]
      
      sample_count <- ncol(gse_data)
      organism <- unique(pData(gse_data)$organism_ch1)[1]
      title <- Meta(gse_data)$title
      summary <- Meta(gse_data)$summary
      platform <- Meta(gse_data)$platform_id
      
      is_rnaseq <- any(grepl("rna.?seq|transcriptome|counts",
                            c(title, summary), ignore.case = TRUE))
      
      return(list(
        accession = gse_accession,
        title = title,
        organism = organism,
        sample_count = sample_count,
        platform = platform,
        is_rnaseq = is_rnaseq,
        summary = substr(summary, 1, 200)
      ))
    }
  }, error = function(e) {
    return(list(accession = gse_accession, error = e$message))
  })
}

# --- Main workflow ---

cell_types <- c(
  "osteoblast",
  "cardiomyocyte",
  "retinal pigment epithelium",
  "neural cell",
  "neuron"
)

# Expanded iPSC terms as requested
ipsc_terms <- c(
  "induced pluripotent stem cell blood",
  "iPSC blood",
  "PBMC iPSC",
  "hematopoietic stem cells",
  "hematopoietic iPSCs",
  "blood derived iPSCs"
)

geo_sqlite_file <- "GEOmetadb.sqlite"

cat("=== Using GEOmetadb local search with RNA-seq filter ===\n")
local_cell_datasets <- search_geo_cell_types(cell_types, min_samples = 5, geo_sqlite_file = geo_sqlite_file)
local_ipsc_datasets <- search_geo_cell_types(ipsc_terms, min_samples = 5, geo_sqlite_file = geo_sqlite_file)

cat("=== Using GEOquery online safe search (RNA-seq only) ===\n")
online_cell_results <- search_with_geoquery_safe(cell_types)
online_ipsc_results <- search_with_geoquery_safe(ipsc_terms)

# Save results
if (nrow(local_cell_datasets) > 0) {
  write.csv(local_cell_datasets, "local_cell_datasets_rnaseq.csv", row.names = FALSE)
  cat("Saved local cell RNA-seq datasets:", nrow(local_cell_datasets), "\n")
}
if (nrow(local_ipsc_datasets) > 0) {
  write.csv(local_ipsc_datasets, "local_ipsc_datasets_rnaseq.csv", row.names = FALSE)
  cat("Saved local iPSC RNA-seq datasets:", nrow(local_ipsc_datasets), "\n")
}

if (nrow(online_cell_results) > 0) {
  write.csv(online_cell_results, "online_cell_datasets_rnaseq.csv", row.names = FALSE)
  cat("Saved online cell RNA-seq datasets:", nrow(online_cell_results), "\n")
}
if (nrow(online_ipsc_results) > 0) {
  write.csv(online_ipsc_results, "online_ipsc_datasets_rnaseq.csv", row.names = FALSE)
  cat("Saved online iPSC RNA-seq datasets:", nrow(online_ipsc_results), "\n")
}

cat("\n=== Script completed ===\n")


=== Using GEOmetadb local search with RNA-seq filter ===
Searching for cell type: osteoblast 
Searching for cell type: cardiomyocyte 
Searching for cell type: retinal pigment epithelium 
Searching for cell type: neural cell 
Searching for cell type: neuron 
Searching for cell type: induced pluripotent stem cell blood 
Searching for cell type: iPSC blood 
Searching for cell type: PBMC iPSC 
Searching for cell type: hematopoietic stem cells 
Searching for cell type: hematopoietic iPSCs 
Searching for cell type: blood derived iPSCs 
=== Using GEOquery online safe search (RNA-seq only) ===
Searching GEO for: osteoblast 
Error searching for osteoblast : object 'destfile' not found 
Searching GEO for: cardiomyocyte 
Error searching for cardiomyocyte : object 'destfile' not found 
Searching GEO for: retinal pigment epithelium 
Error searching for retinal pigment epithelium : object 'destfile' not found 
Searching GEO for: neural cell 
Error searching for neural cell : object 'destfile' not fo